# Model Training

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, recall_score, roc_auc_score, precision_score
import pickle
import json
import os

# Create directories if they don't exist
os.makedirs('models', exist_ok=True)
os.makedirs('submission', exist_ok=True)

## Load and Prepare Data

In [2]:
df = pd.read_csv('data/processed/transformed_dataset.csv')

TARGET = "is_completed"
FEATURES = ["driver_historical_completed_bookings", "driver_historical_acceptance_rate", "trip_distance", "driver_distance", "is_Early_Morning", "is_Morning", "is_Midday", "is_Evening", "is_Night", "is_Late_Night", "is_Monday", "is_Tuesday", "is_Wednesday", "is_Thursday", "is_Friday", "is_Saturday", "is_Sunday"]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Training and Evaluation

In [3]:
models = {
    'logistic_regression': LogisticRegression(random_state=42),
    'decision_tree': DecisionTreeClassifier(random_state=42),
    'random_forest_default': RandomForestClassifier(random_state=42),
    'random_forest_tuned': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
}

metrics = {}
best_model = None
best_roc_auc = -1

In [4]:
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    model_metrics = {
        'f1_score': f1_score(y_test, y_pred),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    }
    
    metrics[name] = model_metrics
    print(f'Metrics for {name}: {model_metrics}')
    
    # Save the model
    with open(f'models/{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
        
    if model_metrics['roc_auc'] > best_roc_auc:
        best_roc_auc = model_metrics['roc_auc']
        best_model = model
        print(f'New best model: {name}')

Training logistic_regression...
Metrics for logistic_regression: {'f1_score': 0.1983608656678192, 'accuracy': 0.5306232114070056, 'precision': 0.4411848476217602, 'recall': 0.12794251259601883, 'roc_auc': 0.5562967169108755}
New best model: logistic_regression
Training decision_tree...
Metrics for decision_tree: {'f1_score': 0.7749116073025438, 'accuracy': 0.7955411704427587, 'precision': 0.7744321619094758, 'recall': 0.7753916467057625, 'roc_auc': 0.7968790731789774}
New best model: decision_tree
Training random_forest_default...
Metrics for random_forest_default: {'f1_score': 0.7310028528139119, 'accuracy': 0.7537271466240112, 'precision': 0.7248781808337845, 'recall': 0.7372319044079183, 'roc_auc': 0.8567097824086614}
New best model: random_forest_default
Training random_forest_tuned...
Metrics for random_forest_tuned: {'f1_score': 0.7829162062016511, 'accuracy': 0.7913422726534285, 'precision': 0.7417105976252648, 'recall': 0.8289694666997054, 'roc_auc': 0.8911693593175068}
New bes

## Save Best Model and Metrics

In [5]:
with open('models/saved_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('submission/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)
    
print('Best model saved to models/saved_model2.pkl')
print('Metrics saved to submission/metrics2.json')

Best model saved to models/saved_model2.pkl
Metrics saved to submission/metrics2.json


In [ ]:
import plotly.graph_objects as go

# Convert metrics dictionary to DataFrame
metrics_df = pd.DataFrame(metrics).T

# Create bar chart
fig = go.Figure()

for metric in metrics_df.columns:
    fig.add_trace(
        go.Bar(
            x=metrics_df.index,
            y=metrics_df[metric],
            name=metric
        )
    )

fig.update_layout(
    title="Model Performance Comparison",
    xaxis_title="Models",
    yaxis_title="Score",
    barmode="group",
    template="plotly"
)


fig.show()


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

for name, model in models.items():
    print(f'\nTraining {name}...')
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    model_metrics = {
        'f1_score': f1_score(y_test, y_pred),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    }
    
    metrics[name] = model_metrics
    print(f'Metrics for {name}: {model_metrics}')
    
    # -----------------------------
    # Per-class performance
    # -----------------------------
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # Save model
    with open(f'models/{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
        
    if model_metrics['roc_auc'] > best_roc_auc:
        best_roc_auc = model_metrics['roc_auc']
        best_model = model
        print(f'New best model: {name}')



Training logistic_regression...
Metrics for logistic_regression: {'f1_score': 0.1983608656678192, 'accuracy': 0.5306232114070056, 'precision': 0.4411848476217602, 'recall': 0.12794251259601883, 'roc_auc': 0.5562967169108755}

Classification Report:
              precision    recall  f1-score   support

           0     0.5442    0.8653    0.6682     43700
           1     0.4412    0.1279    0.1984     36321

    accuracy                         0.5306     80021
   macro avg     0.4927    0.4966    0.4333     80021
weighted avg     0.4974    0.5306    0.4549     80021

Confusion Matrix:
[[37814  5886]
 [31674  4647]]

Training decision_tree...
Metrics for decision_tree: {'f1_score': 0.7749116073025438, 'accuracy': 0.7955411704427587, 'precision': 0.7744321619094758, 'recall': 0.7753916467057625, 'roc_auc': 0.7968790731789774}

Classification Report:
              precision    recall  f1-score   support

           0     0.8131    0.8123    0.8127     43700
           1     0.7744    0